# 03 — Anomaly Detection with Reconstruction Loss

**Goal:** Train an autoencoder on a single 'normal' class. At inference, anything that reconstructs poorly is flagged as anomalous. Evaluate with precision, recall, F1, and ROC-AUC.

**Setup:**
- Normal class: Fashion-MNIST `Sneaker` (label 7)
- Anomalies: every other class
- The model never sees anomalies during training — it just learns what 'sneaker' looks like.

**Author:** Zain Rafeeque

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import (precision_recall_fscore_support, roc_auc_score,
                             roc_curve, confusion_matrix)

tf.random.set_seed(42)
np.random.seed(42)
print('TensorFlow', tf.__version__)

## 1. Build a one-class training set

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = x_train.astype('float32')[..., None] / 255.0
x_test  = x_test.astype('float32')[..., None]  / 255.0

NORMAL_CLASS = 7   # Sneaker
x_train_normal = x_train[y_train == NORMAL_CLASS]
y_test_anomaly = (y_test != NORMAL_CLASS).astype(int)   # 1 = anomaly

print('train (normal only):', x_train_normal.shape)
print('test mix:', x_test.shape, '— anomaly fraction:', y_test_anomaly.mean().round(3))

## 2. Train the convolutional autoencoder on Sneakers only

In [ ]:
def build_ae():
    inp = layers.Input(shape=(28, 28, 1))
    x = layers.Conv2D(16, 3, activation='relu', padding='same')(inp)
    x = layers.MaxPooling2D(2, padding='same')(x)
    x = layers.Conv2D(8, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(2, padding='same')(x)
    x = layers.Conv2DTranspose(8, 3, strides=2, activation='relu', padding='same')(x)
    x = layers.Conv2DTranspose(16, 3, strides=2, activation='relu', padding='same')(x)
    out = layers.Conv2D(1, 3, activation='sigmoid', padding='same')(x)
    return models.Model(inp, out, name='one_class_ae')

ae = build_ae()
ae.compile(optimizer='adam', loss='mse')
history = ae.fit(
    x_train_normal, x_train_normal,
    epochs=15, batch_size=128,
    validation_split=0.1,
    callbacks=[callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=2,
)

## 3. Score the test set

Anomaly score = per-image MSE between input and reconstruction. High score = unusual.

In [ ]:
recon = ae.predict(x_test, batch_size=256, verbose=0)
scores = np.mean((x_test - recon) ** 2, axis=(1, 2, 3))

auc = roc_auc_score(y_test_anomaly, scores)
print(f'ROC-AUC: {auc:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(scores[y_test_anomaly == 0], bins=60, alpha=0.7, label='Normal (sneaker)', density=True)
axes[0].hist(scores[y_test_anomaly == 1], bins=60, alpha=0.7, label='Anomaly (other)', density=True)
axes[0].set_xlabel('reconstruction MSE'); axes[0].set_ylabel('density')
axes[0].set_title('Reconstruction-error distribution'); axes[0].legend(); axes[0].grid(alpha=0.3)

fpr, tpr, _ = roc_curve(y_test_anomaly, scores)
axes[1].plot(fpr, tpr, label=f'AUC = {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC curve'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Pick a threshold and report metrics

Choose the threshold that maximizes F1 on a held-out scoring grid.

In [ ]:
thresholds = np.quantile(scores, np.linspace(0.05, 0.95, 19))
best_f1, best_thr, best_pr = -1, None, None
for thr in thresholds:
    preds = (scores > thr).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_test_anomaly, preds, average='binary', zero_division=0)
    if f1 > best_f1:
        best_f1, best_thr, best_pr = f1, thr, (p, r)

print(f'Best threshold: {best_thr:.5f}')
print(f'Precision: {best_pr[0]:.3f}  Recall: {best_pr[1]:.3f}  F1: {best_f1:.3f}')

preds_best = (scores > best_thr).astype(int)
cm = confusion_matrix(y_test_anomaly, preds_best)
print('\nConfusion matrix [rows=true, cols=pred]')
print(f'           normal  anomaly')
print(f'normal    {cm[0,0]:6d}  {cm[0,1]:7d}')
print(f'anomaly   {cm[1,0]:6d}  {cm[1,1]:7d}')

## 5. What does the model find easy / hard?

Lowest-error samples should be sneakers; highest-error should be visually distant classes.

In [ ]:
CLASS_NAMES = ['T-shirt','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Boot']

low_idx  = np.argsort(scores)[:6]
high_idx = np.argsort(scores)[-6:]

fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for i, k in enumerate(low_idx):
    axes[0, i].imshow(x_test[k, ..., 0], cmap='gray'); axes[0, i].axis('off')
    axes[0, i].set_title(f'{CLASS_NAMES[y_test[k]]}\n{scores[k]:.4f}', fontsize=9)
for i, k in enumerate(high_idx):
    axes[1, i].imshow(x_test[k, ..., 0], cmap='gray'); axes[1, i].axis('off')
    axes[1, i].set_title(f'{CLASS_NAMES[y_test[k]]}\n{scores[k]:.4f}', fontsize=9)
axes[0, 0].set_ylabel('lowest scores\n(most normal)', fontsize=10)
axes[1, 0].set_ylabel('highest scores\n(most anomalous)', fontsize=10)
plt.tight_layout(); plt.show()

## Takeaways

- The model never sees an anomaly during training, yet ROC-AUC is well above the 0.5 random baseline — reconstruction loss is a viable anomaly signal.
- Items visually similar to sneakers (boots, sandals) are the hardest negatives: lower scores → more false negatives.
- For a real production system you'd combine this with a learned threshold per class, drift monitoring, and ideally a small labeled validation set to tune precision/recall trade-offs.